In [1]:
from sedona.spark import SedonaContext
import pyspark.sql.functions as f
import os

In [2]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder().\
    config("spark.executor.memory", "20g").\
    config("spark.driver.memory", "20g").\
    config("spark.memory.offHeap.enabled", True).\
    config("spark.memory.offHeap.size","16g")

sedona = SedonaContext.create(config.getOrCreate())

sedona.sparkContext.setLogLevel("ERROR")

sc = sedona.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/07 18:00:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/07 18:00:36 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/12/07 18:00:36 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/12/07 18:00:36 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/12/07 18:00:36 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/12/07 18:00:36 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/12/07 18:00:36 WARN SimpleFunctionRegistry: The function st_envelop

In [3]:
area_of_analysis = "POLYGON((-87.9401 41.6445, -87.9401 42.0230, -87.5237 42.0230, -87.5237 41.6445, -87.9401 41.6445))"

In [4]:
places = sedona.\
    read.\
    format("geoparquet").\
    load(f"s3a://{bucket_name}/source_data/us_places").\
    where(f"ST_Intersects(geom, ST_GeomFromText('{area_of_analysis}'))")

roads = sedona.\
    read.\
    format("geoparquet").\
    load(f"s3a://{bucket_name}/source_data/us_roads_repartitioned").\
    where(f"ST_Intersects(geometry, ST_GeomFromText('{area_of_analysis}'))")

buildings = sedona.\
    read.\
    format("geoparquet").\
    load(f"s3a://{bucket_name}/source_data/us_buildings").\
    where(f"ST_Intersects(geometry, ST_GeomFromText('{area_of_analysis}'))").\
    where("class='residential'")

categories = sedona.\
    read.\
    parquet(f"s3a://{bucket_name}/source_data/categories_us")

buildings.createOrReplaceTempView("buildings")
roads.createOrReplaceTempView("roads")
places.createOrReplaceTempView("places")
categories.createOrReplaceTempView("categories")

categories.cache()

DataFrame[category_id: string, category_level: int, category_name: string, category_label: string, level1_category_id: string, level1_category_name: string, level2_category_id: string, level2_category_name: string, level3_category_id: string, level3_category_name: string, level4_category_id: string, level4_category_name: string, level5_category_id: string, level5_category_name: string, level6_category_id: string, level6_category_name: string]

# closest road or railway

In [5]:
noisy_road_classes = [
    'motorway',
    'trunk',
    'primary',
    'secondary',
    'tertiary'
]

noisy_roads = roads.where(
    (
        (
            (f.col("class").isin(noisy_road_classes)) &
            (f.col("subtype") == "road")
        ) |
        (
            (f.col("subtype") == "rail") &
            (f.expr("class IS NULL"))
        )
        
    ) 
)

noisy_roads.cache()

noisy_roads.createOrReplaceTempView("noisy_roads")

In [6]:
sedona.sql(
    """
        SELECT 
            b.id,
            ST_DistanceSpheroid(b.geometry, n.geometry) AS distance_to_road_or_railway
        FROM buildings AS b
        JOIN noisy_roads as n ON ST_KNN(b.geometry, n.geometry, 1)
    """
).createOrReplaceTempView("closest_road_or_railway")

In [ ]:
sedona.sql("SELECT * FROM closest_road_or_railway").count()

[Stage 7:===============================>                       (58 + 11) / 100]

# number of grocery stores within 500m

In [ ]:
categories_raw = sedona.sql(
    """
    SELECT 
        category_id
    FROM categories 
    WHERE category_name == 'Grocery Store'
    """
).collect()

categories = [c[0] for c in categories_raw]

In [ ]:
places\
    .where(f.arrays_overlap("fsq_category_ids", f.lit(categories)))\
    .select("fsq_place_id", "geom") \
    .createOrReplaceTempView("grocery_stores")
    
sedona.sql(
    """
    WITH grocery_stores_nearby AS (
        SELECT 
            b.id AS building_id,
            g.fsq_place_id AS g_id 
        FROM buildings AS b
        JOIN grocery_stores AS g ON ST_DWithin(g.geom, b.geometry, 500, true)
    )
    SELECT 
        building_id AS id,
        count(g_id) AS number_of_grocery_stores
    FROM grocery_stores_nearby
    GROUP BY id

    """
).createOrReplaceTempView("grocery_stores_agg")

# Poi categories nearby

In [ ]:
sedona.sql(
    """
    WITH pois_intersected AS (
        SELECT 
            b.id AS building_id,
            p.fsq_place_id AS poi_id,
            p.fsq_category_ids
        FROM buildings AS b
        JOIN places AS p ON ST_DWithin(p.geom, b.geometry, 300, true)
    ),
    pois_with_categories AS (
        SELECT 
            building_id,
            poi_id, 
            explode(fsq_category_ids) AS category_id
        FROM pois_intersected
    ),
    pois_cnt AS (
        SELECT 
            building_id AS id,
            count(poi_id) AS number_of_pois,
            category_id
        FROM pois_with_categories
        GROUP BY id, category_id
    ),
    pois_ranked AS (
        SELECT 
            id,
            ROW_NUMBER() OVER (
                PARTITION BY id ORDER BY number_of_pois DESC
            ) as rank,
            category_id
        FROM pois_cnt
    ),
    rank_3 AS (
        SELECT 
            id,
            category_id
        FROM pois_ranked
        WHERE rank <= 3
    ),
    pois_with_categories_resolved AS (
        SELECT 
            id,
            c.level1_category_name AS category_name
        FROM rank_3 AS p
        JOIN categories AS c ON p.category_id = c.category_id
    )
    SELECT 
        id,
        collect_list(category_name) AS category_names
    FROM pois_with_categories_resolved
    GROUP BY id
    """
).createOrReplaceTempView("poi_categories_nearby")

# Join the data

In [ ]:
sedona.sql("""
    SELECT 
        b.id,
        b.geometry,
        pc.category_names,
        gs.number_of_grocery_stores,
        cr.distance_to_road_or_railway
    FROM buildings AS b
    LEFT JOIN poi_categories_nearby AS pc ON pc.id = b.id
    LEFT JOIN grocery_stores_agg AS gs ON gs.id = b.id
    LEFT JOIN closest_road_or_railway AS cr ON cr.id = b.id
""").show()